# Description

Analyzes the LVs driving the association of Niacin with cardiovascular traits using the ARCHS4 CLAMP model.

Uses the CLAMP-projected SMulTiXcan gene-trait associations and LINCS L1000 drug expression profiles to:
1. Compute per-LV contributions to each drug-disease prediction.
2. Identify LVs commonly driving the Niacin–cardiovascular predictions.
3. Show which LVs are most strongly "affected" by Niacin.

Unlike the PhenoPlier version, there is no tissue selection step because SMulTiXcan already aggregates across 49 GTEx tissues into a single gene-trait z-score matrix.

**Inputs** (from `001-smultixcan-projection.ipynb`):
- `output/drug_disease_analyses/smultixcan-mashr-zscores-projection.pkl`: LVs × traits
- `output/drug_disease_analyses/lincs-projection.pkl`: LVs × drugs

**Output**:
- `output/drug_disease_analyses/analyses/cardiovascular-niacin.h5`

# Module loading

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

from pyprojroot import here

# Settings

In [3]:
# Threshold to select the top-contributing LVs for each selected trait
QUANTILE = 0.95

# Paths

In [4]:
PROJECTIONS_DIR = here('output/drug_disease_analyses')
display(PROJECTIONS_DIR)
assert PROJECTIONS_DIR.exists()

OUTPUT_DIR = PROJECTIONS_DIR / 'analyses'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
display(OUTPUT_DIR)

OUTPUT_FILEPATH = OUTPUT_DIR / 'cardiovascular-niacin.h5'
display(OUTPUT_FILEPATH)

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/analyses')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/analyses/cardiovascular-niacin.h5')

# Data loading

## LINCS projection (LVs × drugs)

In [5]:
lincs_proj = pd.read_pickle(PROJECTIONS_DIR / 'lincs-projection.pkl')
print(f'LINCS projection shape: {lincs_proj.shape}')  # (n_lvs, n_drugs)
display(lincs_proj.head())

LINCS projection shape: (2366, 1170)


perturbagen,DB00014,DB00091,DB00121,DB00130,DB00131,DB00132,DB00136,DB00140,DB00146,DB00150,...,DB08995,DB09002,DB09004,DB09009,DB09010,DB09015,DB09019,DB09020,DB09022,DB09023
LV1,-0.003664,0.021802,0.050683,-0.023411,0.054818,0.022504,-0.016074,-0.007949,-0.012020,-0.022674,...,-0.051500,-0.039418,-0.021987,0.012250,0.009535,0.012377,0.013959,-0.209899,0.026002,0.009327
LV2,0.011797,0.058622,0.001288,-0.023493,0.009715,0.001838,-0.025821,-0.026784,-0.010831,0.000237,...,-0.003300,-0.000286,0.004965,0.011235,0.002291,-0.003366,-0.002029,-0.067440,0.005340,0.011255
LV3,-0.000841,-0.055107,-0.006764,0.027218,-0.012003,-0.010472,-0.021370,-0.007238,-0.005116,0.005954,...,-0.003723,0.009973,-0.007469,0.009228,-0.010771,-0.017754,0.012783,0.066089,0.008973,-0.011303
LV4,0.031992,-0.511423,-0.071332,-0.059371,-0.035587,-0.033645,-0.076674,0.042806,-0.099315,0.031507,...,0.051060,0.022656,0.054687,0.014682,-0.078138,-0.014696,0.015666,-0.238956,-0.018649,-0.047013
LV5,-0.032510,0.123403,0.005948,0.031886,-0.045581,-0.006246,0.092339,0.034634,0.111000,-0.025291,...,-0.020998,-0.019916,-0.030282,0.009627,0.031024,-0.017982,-0.019684,0.164316,0.012215,-0.027974


## SMulTiXcan projection (LVs × traits)

In [6]:
smultixcan_proj = pd.read_pickle(PROJECTIONS_DIR / 'smultixcan-mashr-zscores-projection.pkl')
print(f'SMulTiXcan projection shape: {smultixcan_proj.shape}')  # (n_lvs, n_traits)
display(smultixcan_proj.head())

SMulTiXcan projection shape: (2366, 4091)


,20096_1-Size_of_red_wine_glass_drunk_small_125ml,2345-Ever_had_bowel_cancer_screening,N49-Diagnoses_main_ICD10_N49_Inflammatory_disorders_of_male_genital_organs_not_elsewhere_classified,100011_raw-Iron,5221-Index_of_best_refractometry_result_right,20003_1141150624-Treatmentmedication_code_zomig_25mg_tablet,S69-Diagnoses_main_ICD10_S69_Other_and_unspecified_injuries_of_wrist_and_hand,20024_1136-Job_code_deduced_Information_and_communication_technology_managers,20002_1385-Noncancer_illness_code_selfreported_allergy_or_anaphylactic_reaction_to_food,G6_SLEEPAPNO-Sleep_apnoea,...,Astle_et_al_2016_Sum_basophil_neutrophil_counts,RA_OKADA_TRANS_ETHNIC,pgc.scz2,PGC_ADHD_EUR_2017,MAGIC_FastingGlucose,Astle_et_al_2016_Red_blood_cell_count,SSGAC_Depressive_Symptoms,BCAC_ER_positive_BreastCancer_EUR,IBD.EUR.Inflammatory_Bowel_Disease,Astle_et_al_2016_High_light_scatter_reticulocyte_count
LV1,0.153456,0.171552,0.153334,0.152908,0.134566,0.163124,0.145335,0.146773,0.152961,0.142850,...,0.262381,0.196087,0.286988,0.163989,0.139514,0.285309,0.157414,0.156926,0.194079,0.297641
LV2,0.100974,0.106346,0.101445,0.095510,0.088895,0.089579,0.089118,0.090771,0.082718,0.102523,...,0.172808,0.141204,0.143710,0.121239,0.090788,0.199298,0.095521,0.101541,0.142240,0.164590
LV3,0.070661,0.083573,0.082994,0.065916,0.073920,0.086248,0.073543,0.076102,0.061243,0.069460,...,0.073540,0.090083,0.152892,0.113261,0.060385,0.090092,0.089826,0.072068,0.070305,0.086717
LV4,0.103140,0.111902,0.096378,0.085740,0.107958,0.111800,0.124310,0.087583,0.109811,0.118070,...,0.136074,0.113084,0.168764,0.136986,0.092374,0.133274,0.133228,0.112045,0.119645,0.094503
LV5,0.035550,0.015937,0.014084,-0.014893,0.015324,0.027896,0.024724,0.020440,0.018119,0.027276,...,0.042351,0.049341,0.024278,0.017188,0.025756,0.071969,0.003834,0.013244,0.049691,0.020399


# Niacin and cardiovascular diseases

## Select traits

In [7]:
_phenomexcan_traits = [
    'I70-Diagnoses_main_ICD10_I70_Atherosclerosis',
    'CARDIoGRAM_C4D_CAD_ADDITIVE',
    'I25-Diagnoses_main_ICD10_I25_Chronic_ischaemic_heart_disease',
    '20002_1473-Noncancer_illness_code_selfreported_high_cholesterol',
    '6150_100-Vascularheart_problems_diagnosed_by_doctor_None_of_the_above',
    '6150_1-Vascularheart_problems_diagnosed_by_doctor_Heart_attack',
    'I9_CHD-Major_coronary_heart_disease_event',
    'I9_CORATHER-Coronary_atherosclerosis',
    'I9_IHD-Ischaemic_heart_disease_wide_definition',
    'I9_MI-Myocardial_infarction',
    'I21-Diagnoses_main_ICD10_I21_Acute_myocardial_infarction',
    '20002_1075-Noncancer_illness_code_selfreported_heart_attackmyocardial_infarction',
]

_drug_id = 'DB00627'
_drug_name = 'Niacin'

In [8]:
# Check which traits are present in the SMulTiXcan projection
available_traits = [t for t in _phenomexcan_traits if t in smultixcan_proj.columns]
missing_traits = [t for t in _phenomexcan_traits if t not in smultixcan_proj.columns]
print(f'Available traits: {len(available_traits)} / {len(_phenomexcan_traits)}')
if missing_traits:
    print(f'Missing traits: {missing_traits}')

Available traits: 12 / 12


## Compute Niacin prediction scores for cardiovascular traits

Since we use SMulTiXcan (a single multi-tissue result), there is no tissue selection step.
The prediction score is:
$$\text{score} = -1 \times \mathbf{drug}^T \mathbf{disease}$$
computed in the CLAMP LV space.

In [9]:
# Get Niacin LV vector: (n_lvs,)
drug_data_full = lincs_proj[_drug_id]
print(f'Niacin LV vector shape: {drug_data_full.shape}')
display(drug_data_full.head())

Niacin LV vector shape: (2366,)


LV1    0.018649
LV2    0.023365
LV3   -0.024412
LV4    0.079144
LV5   -0.082024
Name: DB00627, dtype: float64

In [10]:
# Compute Niacin prediction scores for all traits
scores_all = -1.0 * drug_data_full.dot(smultixcan_proj)  # (n_traits,)
print(f'Scores shape: {scores_all.shape}')
display(scores_all.describe())

Scores shape: (4091,)


count    4091.000000
mean       -0.022275
std         0.013662
min        -0.121136
25%        -0.030339
50%        -0.021830
75%        -0.014117
max         0.119824
Name: DB00627, dtype: float64

In [11]:
# Standardize scores
score_mean, score_std = scores_all.mean(), scores_all.std()
scores_all_std = (scores_all - score_mean) / score_std
display(scores_all_std.describe())

count    4.091000e+03
mean     3.560529e-17
std      1.000000e+00
min     -7.236125e+00
25%     -5.902343e-01
50%      3.258651e-02
75%      5.971261e-01
max      1.040103e+01
Name: DB00627, dtype: float64

In [12]:
# Niacin scores for cardiovascular traits (standardized)
drug_df = scores_all_std.loc[available_traits].sort_values()
display(drug_df)

6150_100-Vascularheart_problems_diagnosed_by_doctor_None_of_the_above              -0.752620
I9_CHD-Major_coronary_heart_disease_event                                           0.022594
I9_MI-Myocardial_infarction                                                         0.085474
20002_1075-Noncancer_illness_code_selfreported_heart_attackmyocardial_infarction    0.289668
6150_1-Vascularheart_problems_diagnosed_by_doctor_Heart_attack                      0.503163
I9_CORATHER-Coronary_atherosclerosis                                                0.793512
I25-Diagnoses_main_ICD10_I25_Chronic_ischaemic_heart_disease                        1.006843
CARDIoGRAM_C4D_CAD_ADDITIVE                                                         1.029657
I21-Diagnoses_main_ICD10_I21_Acute_myocardial_infarction                            1.035748
I9_IHD-Ischaemic_heart_disease_wide_definition                                      1.117036
I70-Diagnoses_main_ICD10_I70_Atherosclerosis                          

In [13]:
# Select traits for which Niacin has a high prediction (above global 75th percentile)
q75 = scores_all_std.quantile(0.75)
print(f'Global 75th percentile: {q75:.4f}')
selected_traits = drug_df[drug_df > q75].index.tolist()
print(f'Selected traits: {len(selected_traits)}')
display(selected_traits)

Global 75th percentile: 0.5971
Selected traits: 7


['I9_CORATHER-Coronary_atherosclerosis',
 'I25-Diagnoses_main_ICD10_I25_Chronic_ischaemic_heart_disease',
 'CARDIoGRAM_C4D_CAD_ADDITIVE',
 'I21-Diagnoses_main_ICD10_I21_Acute_myocardial_infarction',
 'I9_IHD-Ischaemic_heart_disease_wide_definition',
 'I70-Diagnoses_main_ICD10_I70_Atherosclerosis',
 '20002_1473-Noncancer_illness_code_selfreported_high_cholesterol']

## Get Niacin LV projection values

In [14]:
drug_data = drug_data_full.copy()
print(f'Niacin drug_data shape: {drug_data.shape}')
display(drug_data.head())

Niacin drug_data shape: (2366,)


LV1    0.018649
LV2    0.023365
LV3   -0.024412
LV4    0.079144
LV5   -0.082024
Name: DB00627, dtype: float64

## Module-based — LVs driving association

Get trait LV vectors from the SMulTiXcan projection (one projection per trait, no tissue selection needed).

In [15]:
# Get LV vectors for selected traits: (n_lvs, n_selected_traits)
module_tissue_data = smultixcan_proj[selected_traits]
print(f'Trait LV data shape: {module_tissue_data.shape}')
display(module_tissue_data.head())

Trait LV data shape: (2366, 7)


,I9_CORATHER-Coronary_atherosclerosis,I25-Diagnoses_main_ICD10_I25_Chronic_ischaemic_heart_disease,CARDIoGRAM_C4D_CAD_ADDITIVE,I21-Diagnoses_main_ICD10_I21_Acute_myocardial_infarction,I9_IHD-Ischaemic_heart_disease_wide_definition,I70-Diagnoses_main_ICD10_I70_Atherosclerosis,20002_1473-Noncancer_illness_code_selfreported_high_cholesterol
LV1,0.180767,0.178314,0.187099,0.180647,0.191821,0.147213,0.243882
LV2,0.098368,0.104393,0.126330,0.109848,0.107834,0.099225,0.118287
LV3,0.108314,0.108590,0.062939,0.095052,0.103996,0.071171,0.093973
LV4,0.141322,0.137770,0.132634,0.125177,0.141706,0.106635,0.143959
LV5,0.040838,0.039030,0.018009,0.025506,0.028560,0.015054,0.003837


In [16]:
# Sanity check: prediction scores should match what we computed earlier
_check = (-1.0 * drug_data.dot(module_tissue_data)).sort_values(ascending=False)
display(_check)

20002_1473-Noncancer_illness_code_selfreported_high_cholesterol    0.017652
I70-Diagnoses_main_ICD10_I70_Atherosclerosis                       0.000271
I9_IHD-Ischaemic_heart_disease_wide_definition                    -0.007014
I21-Diagnoses_main_ICD10_I21_Acute_myocardial_infarction          -0.008125
CARDIoGRAM_C4D_CAD_ADDITIVE                                       -0.008208
I25-Diagnoses_main_ICD10_I25_Chronic_ischaemic_heart_disease      -0.008520
I9_CORATHER-Coronary_atherosclerosis                              -0.011434
Name: DB00627, dtype: float64

In [17]:
# Per-LV contribution: for each trait, how much does each LV contribute to the prediction?
# drug_trait_predictions[lv, trait] = -1 * drug_data[lv] * trait_data[lv]
drug_trait_predictions = pd.DataFrame(
    -1.0 * (drug_data.values[:, None] * module_tissue_data.values),
    columns=module_tissue_data.columns.copy(),
    index=drug_data.index.copy(),
)

print(f'drug_trait_predictions shape: {drug_trait_predictions.shape}')
display(drug_trait_predictions.head())

drug_trait_predictions shape: (2366, 7)


,I9_CORATHER-Coronary_atherosclerosis,I25-Diagnoses_main_ICD10_I25_Chronic_ischaemic_heart_disease,CARDIoGRAM_C4D_CAD_ADDITIVE,I21-Diagnoses_main_ICD10_I21_Acute_myocardial_infarction,I9_IHD-Ischaemic_heart_disease_wide_definition,I70-Diagnoses_main_ICD10_I70_Atherosclerosis,20002_1473-Noncancer_illness_code_selfreported_high_cholesterol
LV1,-0.003371,-0.003325,-0.003489,-0.003369,-0.003577,-0.002745,-0.004548
LV2,-0.002298,-0.002439,-0.002952,-0.002567,-0.002520,-0.002318,-0.002764
LV3,0.002644,0.002651,0.001536,0.002320,0.002539,0.001737,0.002294
LV4,-0.011185,-0.010904,-0.010497,-0.009907,-0.011215,-0.008440,-0.011394
LV5,0.003350,0.003201,0.001477,0.002092,0.002343,0.001235,0.000315


## Get common LVs across selected traits

In [18]:
display(QUANTILE)

0.95

In [19]:
common_lvs = []

for trait in drug_trait_predictions.columns:
    _tmp = drug_trait_predictions[trait]

    # For each trait, get the set of LVs with a positive contribution above the quantile
    _tmp = _tmp[_tmp > 0.0]
    q = _tmp.quantile(QUANTILE)
    _tmp = _tmp[_tmp > q]
    display(f'Trait: {trait}')
    display(f'Number of LVs: {_tmp.shape[0]}')

    _tmp = _tmp.sort_values(ascending=False)
    common_lvs.append(_tmp)

    display(_tmp.head(20))
    print()

'Trait: I9_CORATHER-Coronary_atherosclerosis'

'Number of LVs: 59'

LV866     0.012687
LV146     0.009946
LV36      0.007708
LV705     0.004614
LV5       0.003350
LV134     0.003316
LV52      0.002921
LV10      0.002764
LV12      0.002648
LV3       0.002644
LV156     0.002615
LV86      0.002583
LV149     0.001951
LV678     0.001874
LV2287    0.001862
LV1958    0.001811
LV923     0.001739
LV114     0.001733
LV1530    0.001731
LV84      0.001672
Name: I9_CORATHER-Coronary_atherosclerosis, dtype: float64

'Trait: I25-Diagnoses_main_ICD10_I25_Chronic_ischaemic_heart_disease'

'Number of LVs: 59'

...

'Number of LVs: 61'

LV36      0.004672
LV866     0.004627
LV146     0.003864
LV705     0.003786
LV199     0.002717
LV149     0.002583
LV2337    0.002446
LV86      0.002259
LV2287    0.002163
LV678     0.002152
LV52      0.002141
LV923     0.002115
LV84      0.002095
LV10      0.001869
LV162     0.001821
LV1272    0.001820
LV3       0.001737
LV31      0.001629
LV65      0.001434
LV12      0.001369
Name: I70-Diagnoses_main_ICD10_I70_Atherosclerosis, dtype: float64

'Trait: 20002_1473-Noncancer_illness_code_selfreported_high_cholesterol'

'Number of LVs: 59'

LV705     0.014853
LV866     0.013653
LV156     0.011966
LV36      0.008057
LV678     0.006783
LV2287    0.006568
LV146     0.005872
LV1961    0.005186
LV162     0.004449
LV173     0.004348
LV331     0.004204
LV250     0.003588
LV199     0.003317
LV10      0.003312
LV52      0.003156
LV236     0.002969
LV1668    0.002873
LV114     0.002654
LV3       0.002294
LV1341    0.002182
Name: 20002_1473-Noncancer_illness_code_selfreported_high_cholesterol, dtype: float64

In [20]:
common_lvs_df = (
    pd.concat(common_lvs)
    .reset_index()
    .rename(columns={'index': 'lv', 0: 'value'})
)
display(common_lvs_df.shape)
display(common_lvs_df.head())

(416, 2)

,lv,value
0,LV866,0.012687
1,LV146,0.009946
2,LV36,0.007708
3,LV705,0.004614
4,LV5,0.003350


In [21]:
# Group by LV and sum contributions across traits
lvs_by_sum = common_lvs_df.groupby('lv')['value'].sum().sort_values(ascending=False)
display(lvs_by_sum.head(25))

lv
LV866     0.074583
LV146     0.050845
LV36      0.046530
LV705     0.039434
LV156     0.023349
LV10      0.020801
LV2287    0.019685
LV52      0.018301
LV678     0.017320
LV162     0.016820
LV134     0.016224
LV3       0.015722
LV86      0.015163
LV250     0.015122
LV5       0.013698
LV923     0.013288
LV12      0.012246
LV149     0.011348
LV84      0.011067
LV114     0.010815
LV269     0.010729
LV1530    0.010380
LV199     0.010025
LV1958    0.009631
LV173     0.009528
Name: value, dtype: float64

In [22]:
# Group by LV and count how many traits they appear in
lvs_by_count = common_lvs_df.groupby('lv')['value'].count().sort_values(ascending=False)
display(lvs_by_count.head(25))

lv
LV10      7
LV114     7
LV215     7
LV1923    7
LV162     7
LV146     7
LV12      7
LV923     7
LV86      7
LV705     7
LV783     7
LV502     7
LV2287    7
LV2256    7
LV866     7
LV52      7
LV62      7
LV3       7
LV250     7
LV36      7
LV678     6
LV173     6
LV5       6
LV1830    6
LV1958    6
Name: value, dtype: int64

# Which are the top LVs "affected" by Niacin?

In [23]:
# LVs with largest absolute values in Niacin's projection
drug_data.abs().sort_values(ascending=False).head(30)

LV1485    0.201619
LV1563    0.165625
LV2287    0.146222
LV341     0.136739
LV36      0.128289
LV866     0.117337
LV199     0.113160
LV66      0.109456
LV1272    0.105723
LV1433    0.100992
LV134     0.099038
LV9       0.094707
LV705     0.083049
LV331     0.082300
LV5       0.082024
LV2100    0.080613
LV4       0.079144
LV269     0.075834
LV146     0.073806
LV156     0.072877
LV1961    0.072443
LV107     0.067340
LV86      0.063346
LV380     0.060627
LV267     0.060617
LV1422    0.060288
LV2337    0.060237
LV1341    0.059912
LV671     0.058133
LV138     0.054485
Name: DB00627, dtype: float64

In [24]:
# Top LVs with positive values (Niacin upregulates)
drug_data.sort_values(ascending=False).head(15)

LV1485    0.201619
LV341     0.136739
LV199     0.113160
LV66      0.109456
LV9       0.094707
LV331     0.082300
LV2100    0.080613
LV4       0.079144
LV269     0.075834
LV107     0.067340
LV267     0.060617
LV1422    0.060288
LV1341    0.059912
LV671     0.058133
LV85      0.054465
Name: DB00627, dtype: float64

In [25]:
# Top LVs with negative values (Niacin downregulates)
drug_data.sort_values(ascending=True).head(15)

LV1563   -0.165625
LV2287   -0.146222
LV36     -0.128289
LV866    -0.117337
LV1272   -0.105723
LV1433   -0.100992
LV134    -0.099038
LV705    -0.083049
LV5      -0.082024
LV146    -0.073806
LV156    -0.072877
LV1961   -0.072443
LV86     -0.063346
LV380    -0.060627
LV2337   -0.060237
Name: DB00627, dtype: float64

# Save

In [26]:
with pd.HDFStore(OUTPUT_FILEPATH, mode='w', complevel=4) as store:
    store.put('traits_module_tissue_data', module_tissue_data, format='fixed')
    store.put('drug_data', drug_data, format='fixed')
    store.put('drug_trait_predictions', drug_trait_predictions, format='fixed')
    store.put('common_lvs', common_lvs_df, format='fixed')

print(f'Saved to: {OUTPUT_FILEPATH}')

Saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/analyses/cardiovascular-niacin.h5
